In [5]:
import sys
import os
os.chdir("/home/felixpersson/MasterThesisProject")
print(os.getcwd())

/home/felixpersson/MasterThesisProject


In [6]:
import numpy as np
import json

import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from matplotlib.ticker import LogLocator
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator, LogFormatterMathtext, NullFormatter

from pathlib import Path

from utils.solver import Solver
from utils.iso_reactor_utils import generate_config, generate_config_seq
from utils.interpolator import OpenMCTallyGridSurrogate

from coupled.vapour_reactor import VapourReactor
from coupled.iso_reactor import Reactor

from models.moderator.triangle_mesh import UnstructuredMesh, Surface
from models.moderator.mesh_conduction_model import ModeratorDiscretisedMesh
from models.moderator.mesh_conduction_numba_model import ModeratorDiscretisedMeshFast

from utils.material_properties import moderator_k


In [7]:
with open("./data/reactor_data.json", "r") as f:
    data = json.load(f)
    data["Reactor"]["power"]["thermal"] = 8e6
    data["HeatPipe"]["material"]["h_cond"] = 320

In [8]:
# Ns = [35, 30, 60]
Ns = [15, 15, 60]
Ns_coarse = [15, 15, 30]
cfg = generate_config(data, *Ns)
cfg_coarse = generate_config(data, *Ns_coarse)

iso_reactor        = Reactor(cfg)
vap_reactor        = VapourReactor(cfg)
vap_reactor_coarse = VapourReactor(cfg_coarse)
vap_reactor_full   = VapourReactor(cfg)

vap_reactor.heatpipe.solid.k_temperature = 1100

static_T = 900
iso_reactor.set_variable_neutron_data(False, static_T)
vap_reactor.set_variable_neutron_data(False, static_T)
vap_reactor_coarse.set_variable_k(True)
vap_reactor_coarse.set_variable_neutron_data(True)
vap_reactor_full.set_variable_k(True)
vap_reactor_full.set_variable_neutron_data(True)

solver_iso      = Solver([iso_reactor])
solver_vap      = Solver([vap_reactor_coarse, vap_reactor], iterate = True)
solver_vap_full = Solver([vap_reactor_coarse, vap_reactor_full], iterate = True)
solver_iso.fsolve()
solver_vap.fsolve()
solver_vap_full.fsolve()

Mesh warning: N_Z changed from 60 to 69
Mesh warning: Fuel pin N_Z changed from 27 to 45
Mesh warning: Neutronics N_Z changed from 27 to 45
Mesh warning: N_Z changed from 30 to 23
Mesh warning: Fuel pin N_Z changed from 13 to 15
Mesh warning: Neutronics N_Z changed from 13 to 15
fsolve message: The solution converged.
Solving Heat pipe initial values
fsolve message: The solution converged.
Running iteration 1 ...
fsolve message: The solution converged.
Running iteration 2 ...
fsolve message: The solution converged.
Solving Heat pipe initial values
fsolve message: The solution converged.
Running iteration 1 ...
fsolve message: The solution converged.
Running iteration 2 ...
fsolve message: The solution converged.


In [9]:
import numpy as np
import meshio
import pyvista as pv

# Import these from wherever they currently live in your project.
# from models.moderator.unstructured_mesh import UnstructuredMesh
# from models.moderator.moderator_discretised_mesh_fast import ModeratorDiscretisedMeshFast


# -------------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------------

def edges_from_centres_and_half_widths(x, dx_m, dx_p):
    """
    Reconstruct cell edges from cell centres and minus/plus half-widths.
    Works for non-uniform radial meshes.
    """
    x = np.asarray(x, dtype=float)
    dx_m = np.asarray(dx_m, dtype=float)
    dx_p = np.asarray(dx_p, dtype=float)

    edges = np.empty(x.size + 1, dtype=float)
    edges[:-1] = x - dx_m
    edges[1:] = x + dx_p
    return edges


# -------------------------------------------------------------------------
# Plotting scale
# -------------------------------------------------------------------------

# Plotting-only compression of the axial direction.
# 1.0  = true geometry
# 0.05 = show axial length at 5% of physical length
AXIAL_SCALE = 0.04


def annular_sector_grid(
    center_xy,
    r_edges,
    z_edges,
    T_zr,
    *,
    theta0=np.deg2rad(25.0),
    theta1=np.deg2rad(335.0),
    n_theta=72,
    name="T",
    axial_scale=1.0,
):
    """
    Creates a cut-away annular/cylindrical sector with cell temperatures.

    The axial coordinate is scaled for plotting only.
    """
    cx, cy = center_xy

    z_edges_plot = np.asarray(z_edges, dtype=float) * axial_scale

    theta_edges = np.linspace(theta0, theta1, n_theta)
    theta, z, r = np.meshgrid(theta_edges, z_edges_plot, r_edges, indexing="ij")

    x = cx + r * np.cos(theta)
    y = cy + r * np.sin(theta)

    grid = pv.StructuredGrid(x, y, z)

    T_cells = np.broadcast_to(
        T_zr[None, :, :],
        (n_theta - 1, T_zr.shape[0], T_zr.shape[1]),
    )

    grid.cell_data[name] = T_cells.ravel(order="F")
    return grid


def extrude_triangular_mesh_slice(
    points_2d,
    triangles,
    z0,
    z1,
    cell_T,
    name="T",
    axial_scale=1.0,
):
    """
    Extrudes a 2D triangular moderator mesh into one wedge-cell-thick 3D slab.

    The axial coordinate is scaled for plotting only.
    """
    n_points = points_2d.shape[0]

    z0_plot = z0 * axial_scale
    z1_plot = z1 * axial_scale

    points_bottom = np.column_stack(
        [points_2d[:, 0], points_2d[:, 1], np.full(n_points, z0_plot)]
    )
    points_top = np.column_stack(
        [points_2d[:, 0], points_2d[:, 1], np.full(n_points, z1_plot)]
    )

    points_3d = np.vstack([points_bottom, points_top])

    cells = []
    for tri in triangles:
        p0, p1, p2 = tri
        cells.extend([
            6,
            p0, p1, p2,
            p0 + n_points, p1 + n_points, p2 + n_points,
        ])

    cell_types = np.full(triangles.shape[0], pv.CellType.WEDGE, dtype=np.uint8)

    grid = pv.UnstructuredGrid(
        np.asarray(cells, dtype=np.int64),
        cell_types,
        points_3d,
    )

    grid.cell_data[name] = np.asarray(cell_T, dtype=float)
    return grid

def annular_end_cap_grid(
    center_xy,
    r_edges,
    z,
    scalar_value,
    *,
    theta0,
    theta1,
    n_theta,
    name="T",
    axial_scale=1.0,
    z_nudge=0.0,
):
    """
    Creates a flat circular/annular end cap with one uniform scalar value.
    z_nudge is in plotted coordinates, i.e. after axial scaling.
    """
    cx, cy = center_xy

    theta_edges = np.linspace(theta0, theta1, n_theta)
    r_edges = np.asarray(r_edges, dtype=float)

    z_plot = z * axial_scale + z_nudge

    points = []
    for theta in theta_edges:
        for r in r_edges:
            points.append([
                cx + r * np.cos(theta),
                cy + r * np.sin(theta),
                z_plot,
            ])

    points = np.asarray(points, dtype=float)

    n_r = len(r_edges)
    faces = []

    for it in range(len(theta_edges) - 1):
        for ir in range(n_r - 1):
            p00 = it * n_r + ir
            p01 = it * n_r + ir + 1
            p11 = (it + 1) * n_r + ir + 1
            p10 = (it + 1) * n_r + ir

            faces.extend([4, p00, p01, p11, p10])

    cap = pv.PolyData(points, np.asarray(faces, dtype=np.int64)).triangulate()
    cap.cell_data[name] = np.full(cap.n_cells, scalar_value, dtype=float)

    return cap

In [10]:


# -------------------------------------------------------------------------
# Extract solved reactor fields
# -------------------------------------------------------------------------

# From your existing code:
((T_solid_vap_full, u_v_full, T_v_full), T_FP_vap_full, (phi_n_g_vap_full, k_vap_full)) = solver_vap_full.solutions[1]

hp_solid_model = vap_reactor_full.heatpipe.solid
fp_model = vap_reactor_full.fuel_pin_thermal_model

# Make sure discretisation arrays exist.
_, hp_delta_Rp, hp_delta_Rm, hp_delta_Z_half = hp_solid_model._initialize_discretization()
fp_model.initialize_discretization()

# Heat-pipe edges.
hp_r_edges = edges_from_centres_and_half_widths(
    hp_solid_model.R,
    hp_delta_Rm,
    hp_delta_Rp,
)

hp_z_edges = edges_from_centres_and_half_widths(
    hp_solid_model.Z,
    hp_delta_Z_half,
    hp_delta_Z_half,
)

# Fuel-pin edges.
fp_r_edges = edges_from_centres_and_half_widths(
    fp_model.R,
    fp_model.delta_Rm,
    fp_model.delta_Rp,
)

fp_z_edges = np.linspace(
    0.0,
    fp_model.cfg.geometry.l,
    fp_model.cfg.mesh.N_Z + 1,
)

# Middle fuel-pin / evaporator cell.
iz_fp_mid = fp_model.cfg.mesh.N_Z // 2
z0_mid = fp_z_edges[iz_fp_mid]
z1_mid = fp_z_edges[iz_fp_mid + 1]
z_mid = 0.5 * (z0_mid + z1_mid)

# Closest heat-pipe evaporator cell to the same axial location.
iz_hp_mid = np.argmin(
    np.abs(hp_solid_model.Z[:vap_reactor_full.cfg_HP.mesh.N_evap] - z_mid)
)

T_HP_outer_mid = float(T_solid_vap_full[iz_hp_mid, -1])
T_FP_outer_mid = float(T_FP_vap_full[iz_fp_mid, -1])

print(f"Middle FP cell index: {iz_fp_mid}")
print(f"Moderator slice z-range: [{z0_mid:.5f}, {z1_mid:.5f}] m")
print(f"T_HP_outer_mid = {T_HP_outer_mid:.2f} K")
print(f"T_FP_outer_mid = {T_FP_outer_mid:.2f} K")


# -------------------------------------------------------------------------
# Solve one moderator slice using hex_mesh.msh
# -------------------------------------------------------------------------

mesh = meshio.read("models/moderator/hex_mesh.msh")

points = mesh.points[:, :2]
triangles = mesh.cells_dict["triangle"]

mod_mesh_2d = UnstructuredMesh(points, triangles)

# -------------------------------------------------------------------------
# Geometry layout matching create_hexagonal_reactor_mesh(...)
# -------------------------------------------------------------------------

theta_hex = np.pi / 6.0

# IMPORTANT:
# This must match the data used when generating models/moderator/hex_mesh.msh.
# In your mesh-generation script this is called either `pin_cell_pitch`
# or `l_pitch`.
pin_pitch = 0.032  # or: data["Reactor"]["geometry"]["pin_cell_pitch"] if you store it there

# Use the same radii that were used to generate the mesh.
# These should match the holes in hex_mesh.msh.
r_HP_mod = 0.011
r_f_mod = 0.0072

# Heat-pipe centres in the 1/12 slice.
HP_centers = [
    [0.0, 0.0],
    [np.sqrt(3.0) / 2.0 * pin_pitch, 1.5 * pin_pitch],
]

# Fuel-pin centres in the 1/12 slice.
fuel_pin_centers = [
    [0.0, 1.0 * pin_pitch],
    [0.0, 2.0 * pin_pitch],
    [np.sqrt(3.0) / 2.0 * pin_pitch, 2.5 * pin_pitch],
]

# Keep your existing Neumann fuel-pin formulation.
# The important change is that T_HP and T_FP come from the actual middle reactor cell.
data_mod_mid = {
    "cell_k": 20.0 * np.ones(triangles.shape[0]),
    "cell_T": np.ones(triangles.shape[0]),

    # Same style as your existing moderator solve.
    # If you later want this to be exactly consistent with the solved axial power,
    # replace this with a q' derived from the middle-cell fuel power.
    "Q_in": 15e6 / 2970 / 1.6 * 2.0,

    "HP_centers": HP_centers,
    "fuel_pin_centers": fuel_pin_centers,

    "HP_radius": r_HP_mod,
    "fuel_pin_radius": r_f_mod,

    "T_HP": T_HP_outer_mid,
    "T_FP": T_FP_outer_mid,

    "HP_BC": "Dirichlet",
    "FP_BC": "vonNeumann",
}

mod_mesh_mid = ModeratorDiscretisedMeshFast(
    mesh=mod_mesh_2d,
    data=data_mod_mid,
)

T_mod_mid = mod_mesh_mid.solve_nonlinear_picard(
    xtol=1e-10,
    maxfev=10000,
    verbose=True,
    use_linear_guess=True,
    omega=0.5,
    max_outer_iter=200,
)


Middle FP cell index: 22
Moderator slice z-range: [0.73333, 0.76667] m
T_HP_outer_mid = 1015.27 K
T_FP_outer_mid = 1061.84 K

Picard  1 | NK: ||R||_inf = 1.499724e-02, ||R||_2 = 2.985140e-01
Picard  1 | NK: ||R||_inf = 3.725689e-03, ||R||_2 = 1.568029e-01
Picard  1 | NK: ||R||_inf = 2.385475e-03, ||R||_2 = 1.264113e-01
Picard  1 | NK: ||R||_inf = 3.249095e-03, ||R||_2 = 1.153512e-01
Picard  1 | NK: ||R||_inf = 2.174902e-03, ||R||_2 = 9.899616e-02
Picard  1 | NK: ||R||_inf = 2.363362e-03, ||R||_2 = 8.013036e-02
Picard  1 | NK: ||R||_inf = 2.006514e-03, ||R||_2 = 5.288314e-02
Picard  1 | NK: ||R||_inf = 8.942170e-04, ||R||_2 = 3.304098e-02
Picard  1 | NK: ||R||_inf = 7.134897e-04, ||R||_2 = 2.439852e-02
Picard  1 | NK: ||R||_inf = 5.732608e-04, ||R||_2 = 1.674255e-02
Picard  1 | NK: ||R||_inf = 3.835913e-04, ||R||_2 = 1.135547e-02
Picard  1 | NK: ||R||_inf = 1.896382e-04, ||R||_2 = 6.030064e-03
Picard  1 | NK: ||R||_inf = 8.746176e-05, ||R||_2 = 2.339650e-03
Picard  1 | NK: ||R||_inf = 4

In [11]:
import pyvista as pv

try:
    pv.close_all()
except Exception:
    pass

pv.set_jupyter_backend("none")
pv.OFF_SCREEN = False

In [ ]:
import numpy as np
import pyvista as pv


# -------------------------------------------------------------------------
# Geometry layout matching create_hexagonal_reactor_mesh(...)
# -------------------------------------------------------------------------

pin_pitch = 0.032   # Must match the mesh-generation geometry
r_HP_mod  = 0.011   # Must match hex_mesh.msh
r_f_mod   = 0.0072  # Must match hex_mesh.msh

HP_centers = [
    [0.0, 0.0],
    [np.sqrt(3.0) / 2.0 * pin_pitch, 1.5 * pin_pitch],
]

fuel_pin_centers = [
    [0.0, 1.0 * pin_pitch],
    [0.0, 2.0 * pin_pitch],
    [np.sqrt(3.0) / 2.0 * pin_pitch, 2.5 * pin_pitch],
]


# -------------------------------------------------------------------------
# Plotting-only visual scaling
# -------------------------------------------------------------------------

# Compress axial dimension for visual balance
AXIAL_SCALE = 0.04

# Match fuel-pin radius to moderator hole
FP_RADIAL_SCALE = r_f_mod / fp_r_edges[-1]

# Match heat-pipe radius to moderator hole, then shrink slightly to avoid
# overlap / visual artifacts against moderator hole boundary
HP_RADIAL_SCALE = r_HP_mod / hp_r_edges[-1]
HEAT_PIPE_RADIUS_CLEARANCE = 0.995

fp_r_edges_plot = fp_r_edges * FP_RADIAL_SCALE
hp_r_edges_plot = hp_r_edges * HP_RADIAL_SCALE * HEAT_PIPE_RADIUS_CLEARANCE

# Keep vapour slightly inside wick inner boundary
VAPOUR_RADIUS_CLEARANCE = 0.995
vapour_radius_plot = hp_r_edges_plot[0] * VAPOUR_RADIUS_CLEARANCE


# -------------------------------------------------------------------------
# Angular cut-away controls
# -------------------------------------------------------------------------

# Rotated 180 degrees relative to the original cut
CUT_THETA0 = np.deg2rad(-155.0)
CUT_THETA1 = np.deg2rad(155.0)

N_THETA = 120


# -------------------------------------------------------------------------
# Optional moderator temperature shift
# -------------------------------------------------------------------------

T_mod_mid_plot = T_mod_mid.copy()

# Only keep this if your moderator solver has the same offset issue
# as in your current plotting snippet.
T_mod_mid_plot -= np.min(T_mod_mid_plot) - T_HP_outer_mid


# -------------------------------------------------------------------------
# Build 3D objects
# -------------------------------------------------------------------------

body_objects = []
cap_objects = []

# Use a larger plotted-coordinate offset than 1e-5.
# This is still visually tiny, but large enough to avoid z-buffer fighting.
CAP_NUDGE = 0.00003 * max(
    fp_z_edges[-1] - fp_z_edges[0],
    hp_z_edges[-1] - hp_z_edges[0],
) * AXIAL_SCALE

print(f"CAP_NUDGE = {CAP_NUDGE:.3e}")

# Moderator: one actual triangular mesh slice of thickness Delta_Z
mod_grid = extrude_triangular_mesh_slice(
    points,
    triangles,
    z0_mid,
    z1_mid,
    T_mod_mid_plot,
    axial_scale=AXIAL_SCALE,
)
body_objects.append(("moderator", mod_grid))


# -------------------------------------------------------------------------
# Heat-pipe solid + vapour at each heat-pipe center
# -------------------------------------------------------------------------

vapour_r_edges = np.array([0.0, vapour_radius_plot])

for hp_center in HP_centers:
    # Wall/wick body
    hp_solid_grid = annular_sector_grid(
        center_xy=hp_center,
        r_edges=hp_r_edges_plot,
        z_edges=hp_z_edges,
        T_zr=T_solid_vap_full,
        theta0=CUT_THETA0,
        theta1=CUT_THETA1,
        n_theta=N_THETA,
        axial_scale=AXIAL_SCALE,
    )
    body_objects.append(("heat-pipe wall/wick", hp_solid_grid))

    # Wall/wick bottom cap: outer wall temperature at first axial cell
    hp_bottom_cap = annular_end_cap_grid(
        center_xy=hp_center,
        r_edges=hp_r_edges_plot,
        z=hp_z_edges[0],
        scalar_value=float(T_solid_vap_full[0, -1]),
        theta0=CUT_THETA0,
        theta1=CUT_THETA1,
        n_theta=N_THETA,
        axial_scale=AXIAL_SCALE,
        z_nudge=-CAP_NUDGE,
    )
    cap_objects.append(("heat-pipe wall/wick", hp_bottom_cap))

    # Wall/wick top cap: outer wall temperature at last axial cell
    hp_top_cap = annular_end_cap_grid(
        center_xy=hp_center,
        r_edges=hp_r_edges_plot,
        z=hp_z_edges[-1],
        scalar_value=float(T_solid_vap_full[-1, -1]),
        theta0=CUT_THETA0,
        theta1=CUT_THETA1,
        n_theta=N_THETA,
        axial_scale=AXIAL_SCALE,
        z_nudge=CAP_NUDGE,
    )
    cap_objects.append(("heat-pipe wall/wick", hp_top_cap))

    # Vapour body
    vapour_grid = annular_sector_grid(
        center_xy=hp_center,
        r_edges=vapour_r_edges,
        z_edges=hp_z_edges,
        T_zr=T_v_full[:, None],
        theta0=CUT_THETA0,
        theta1=CUT_THETA1,
        n_theta=N_THETA,
        axial_scale=AXIAL_SCALE,
    )
    body_objects.append(("heat-pipe vapour", vapour_grid))

    # Vapour bottom cap
    vap_bottom_cap = annular_end_cap_grid(
        center_xy=hp_center,
        r_edges=vapour_r_edges,
        z=hp_z_edges[0],
        scalar_value=float(T_v_full[0]),
        theta0=CUT_THETA0,
        theta1=CUT_THETA1,
        n_theta=N_THETA,
        axial_scale=AXIAL_SCALE,
        z_nudge=-1.5 * CAP_NUDGE,
    )
    cap_objects.append(("heat-pipe vapour", vap_bottom_cap))

    # Vapour top cap
    vap_top_cap = annular_end_cap_grid(
        center_xy=hp_center,
        r_edges=vapour_r_edges,
        z=hp_z_edges[-1],
        scalar_value=float(T_v_full[-1]),
        theta0=CUT_THETA0,
        theta1=CUT_THETA1,
        n_theta=N_THETA,
        axial_scale=AXIAL_SCALE,
        z_nudge=1.5 * CAP_NUDGE,
    )
    cap_objects.append(("heat-pipe vapour", vap_top_cap))


# -------------------------------------------------------------------------
# Fuel pins at each fuel-pin center
# -------------------------------------------------------------------------

for fp_center in fuel_pin_centers:
    fp_grid = annular_sector_grid(
        center_xy=fp_center,
        r_edges=fp_r_edges_plot,
        z_edges=fp_z_edges,
        T_zr=T_FP_vap_full,
        theta0=CUT_THETA0,
        theta1=CUT_THETA1,
        n_theta=N_THETA,
        axial_scale=AXIAL_SCALE,
    )
    body_objects.append(("fuel pin", fp_grid))

    # Bottom cap: fuel-pin outer wall temperature at first axial cell
    fp_bottom_cap = annular_end_cap_grid(
        center_xy=fp_center,
        r_edges=fp_r_edges_plot,
        z=fp_z_edges[0],
        scalar_value=float(T_FP_vap_full[0, -1]),
        theta0=CUT_THETA0,
        theta1=CUT_THETA1,
        n_theta=N_THETA,
        axial_scale=AXIAL_SCALE,
        z_nudge=-CAP_NUDGE,
    )
    cap_objects.append(("fuel pin", fp_bottom_cap))

    # Top cap: fuel-pin outer wall temperature at last axial cell
    fp_top_cap = annular_end_cap_grid(
        center_xy=fp_center,
        r_edges=fp_r_edges_plot,
        z=fp_z_edges[-1],
        scalar_value=float(T_FP_vap_full[-1, -1]),
        theta0=CUT_THETA0,
        theta1=CUT_THETA1,
        n_theta=N_THETA,
        axial_scale=AXIAL_SCALE,
        z_nudge=CAP_NUDGE,
    )
    cap_objects.append(("fuel pin", fp_top_cap))


# Combined list for temperature scaling
objects = body_objects + cap_objects


# -------------------------------------------------------------------------
# Render in native PyVista viewer window
# Separate cmaps for:
#   1) fuel pin + moderator + heat-pipe wall/wick
#   2) heat-pipe vapour
# -------------------------------------------------------------------------

pv.set_jupyter_backend("none")
pv.OFF_SCREEN = False

SOLID_LABELS = {"fuel pin", "moderator", "heat-pipe wall/wick"}
VAPOUR_LABELS = {"heat-pipe vapour"}

SOLID_CMAP = "inferno"
VAPOUR_CMAP = "Blues"

solid_temperatures = np.concatenate([
    obj.cell_data["T"]
    for label, obj in objects
    if label in SOLID_LABELS
])

vapour_temperatures = np.concatenate([
    obj.cell_data["T"]
    for label, obj in objects
    if label in VAPOUR_LABELS
])

solid_clim = [
    float(np.nanmin(solid_temperatures)),
    float(np.nanmax(solid_temperatures)),
]

vapour_clim = [
    float(np.nanmin(vapour_temperatures)),
    float(np.nanmax(vapour_temperatures)),
]

print("Solid temperature range (FP + moderator + wall/wick):", solid_clim)
print("Vapour temperature range:", vapour_clim)

plotter = pv.Plotter(
    notebook=False,
    window_size=(1800, 1200),
)

plotter.set_background("white")
plotter.disable_depth_peeling()

# Render bodies first
for label, obj in body_objects:
    if label in VAPOUR_LABELS:
        cmap = VAPOUR_CMAP
        clim = vapour_clim
    else:
        cmap = SOLID_CMAP
        clim = solid_clim

    plotter.add_mesh(
        obj,
        scalars="T",
        clim=clim,
        cmap=cmap,
        opacity=1.0,
        show_edges=False,
        smooth_shading=True,
        show_scalar_bar=False,
        pickable=True,
        lighting=False,
    )

# Render caps last so they visually cover the original axial end faces
for label, obj in cap_objects:
    if label in VAPOUR_LABELS:
        cmap = VAPOUR_CMAP
        clim = vapour_clim
    else:
        cmap = SOLID_CMAP
        clim = solid_clim

    plotter.add_mesh(
        obj,
        scalars="T",
        clim=clim,
        cmap=cmap,
        opacity=1.0,
        show_edges=False,
        smooth_shading=False,
        show_scalar_bar=False,
        pickable=False,
        lighting=False,
    )

# Draw moderator outline last
plotter.add_mesh(
    mod_grid.extract_feature_edges(),
    color="black",
    line_width=0.8,
    opacity=1.0,
    show_scalar_bar=False,
    pickable=False,
)

plotter.view_isometric()

try:
    plotter.show(
        interactive=True,
        auto_close=False,
    )
finally:
    plotter.close()
    pv.close_all()

CAP_NUDGE = 2.760e-06
Solid temperature range (FP + moderator + wall/wick): [994.890918946376, 1143.2730343555445]
Vapour temperature range: [1004.0800687658906, 1010.6603547283879]


MESA: error: ZINK: failed to choose pdev
glx: failed to create drisw screen
MESA: error: ZINK: failed to choose pdev
glx: failed to create drisw screen
MESA: error: ZINK: failed to choose pdev
glx: failed to create drisw screen


In [ ]:
import numpy as np
import pyvista as pv


# -------------------------------------------------------------------------
# Geometry layout matching create_hexagonal_reactor_mesh(...)
# -------------------------------------------------------------------------

pin_pitch = 0.032   # Must match the mesh-generation geometry
r_HP_mod  = 0.011   # Must match hex_mesh.msh
r_f_mod   = 0.0072  # Must match hex_mesh.msh

# Original 1/12-slice centres from create_hexagonal_reactor_mesh(...)
HP_centers_slice = [
    [0.0, 0.0],
    [np.sqrt(3.0) / 2.0 * pin_pitch, 1.5 * pin_pitch],
]

fuel_pin_centers_slice = [
    [0.0, 1.0 * pin_pitch],
    [0.0, 2.0 * pin_pitch],
    [np.sqrt(3.0) / 2.0 * pin_pitch, 2.5 * pin_pitch],
]


# -------------------------------------------------------------------------
# Plotting-only visual scaling
# -------------------------------------------------------------------------

AXIAL_SCALE = 0.04

FP_RADIAL_SCALE = r_f_mod / fp_r_edges[-1]

HP_RADIAL_SCALE = r_HP_mod / hp_r_edges[-1]
HEAT_PIPE_RADIUS_CLEARANCE = 0.995

fp_r_edges_plot = fp_r_edges * FP_RADIAL_SCALE
hp_r_edges_plot = hp_r_edges * HP_RADIAL_SCALE * HEAT_PIPE_RADIUS_CLEARANCE

VAPOUR_RADIUS_CLEARANCE = 0.995
vapour_radius_plot = hp_r_edges_plot[0] * VAPOUR_RADIUS_CLEARANCE
vapour_r_edges = np.array([0.0, vapour_radius_plot])


# -------------------------------------------------------------------------
# Angular cut-away controls
# -------------------------------------------------------------------------

# Full cylinders for most of the assembly
FULL_THETA0 = 0.0
FULL_THETA1 = 2.0 * np.pi

# Cut-away cylinders for one selected 1/6 sector
# This removes a wedge centred around 180 degrees in the local cylinder view.
CUT_THETA0 = np.deg2rad(-245.0)
CUT_THETA1 = np.deg2rad(65.0)

# Only fuel pins / heat pipes whose centre lies inside this 60-degree sector
# are rendered with a cut-away opening.
CUTAWAY_SECTOR_CENTER_DEG = 90.0
CUTAWAY_SECTOR_WIDTH_DEG = 60.0

N_THETA_CUT = 120
N_THETA_FULL = 180


def angle_difference_deg(a, b):
    """
    Smallest signed angular difference a - b in degrees.
    """
    return (a - b + 180.0) % 360.0 - 180.0


def is_in_cutaway_sector(center_xy):
    """
    Returns True if the component centre belongs to the selected 1/6 sector.
    """
    x, y = center_xy

    # The central component has no angular sector. Keep it closed.
    if np.hypot(x, y) < 1e-12:
        return False

    angle = np.rad2deg(np.arctan2(y, x)) % 360.0

    return abs(
        angle_difference_deg(angle, CUTAWAY_SECTOR_CENTER_DEG)
    ) <= CUTAWAY_SECTOR_WIDTH_DEG / 2.0


def theta_settings_for_center(center_xy):
    """
    Use cut-away rendering only for one 1/6 sector.
    Use full 360-degree rendering elsewhere.
    """
    if is_in_cutaway_sector(center_xy):
        return CUT_THETA0, CUT_THETA1, N_THETA_CUT
    else:
        return FULL_THETA0, FULL_THETA1, N_THETA_FULL


# -------------------------------------------------------------------------
# Optional moderator temperature shift
# -------------------------------------------------------------------------

T_mod_mid_plot = T_mod_mid.copy()

# Only keep this if your moderator solver has the same offset issue.
T_mod_mid_plot -= np.min(T_mod_mid_plot) - T_HP_outer_mid


# -------------------------------------------------------------------------
# End-cap helper
# -------------------------------------------------------------------------

def annular_end_cap_grid(
    center_xy,
    r_edges,
    z,
    scalar_value,
    *,
    theta0,
    theta1,
    n_theta,
    name="T",
    axial_scale=1.0,
    z_nudge=0.0,
):
    """
    Creates a flat circular/annular end cap with one uniform scalar value.

    z_nudge is in plotted coordinates, i.e. after axial scaling.
    """
    cx, cy = center_xy

    theta_edges = np.linspace(theta0, theta1, n_theta)
    r_edges = np.asarray(r_edges, dtype=float)

    z_plot = z * axial_scale + z_nudge

    points_cap = []
    for theta in theta_edges:
        for r in r_edges:
            points_cap.append([
                cx + r * np.cos(theta),
                cy + r * np.sin(theta),
                z_plot,
            ])

    points_cap = np.asarray(points_cap, dtype=float)

    n_r = len(r_edges)
    faces = []

    for it in range(len(theta_edges) - 1):
        for ir in range(n_r - 1):
            p00 = it * n_r + ir
            p01 = it * n_r + ir + 1
            p11 = (it + 1) * n_r + ir + 1
            p10 = (it + 1) * n_r + ir

            faces.extend([4, p00, p01, p11, p10])

    cap = pv.PolyData(points_cap, np.asarray(faces, dtype=np.int64)).triangulate()
    cap.cell_data[name] = np.full(cap.n_cells, scalar_value, dtype=float)

    return cap


# -------------------------------------------------------------------------
# Symmetry helpers: 1/12 slice -> full hexagon
# -------------------------------------------------------------------------

def transform_xy(point_xy, *, reflect_x=False, rotate_deg=0.0):
    """
    Reflect a 2D point across x=0 and/or rotate around the origin.
    """
    x, y = float(point_xy[0]), float(point_xy[1])

    if reflect_x:
        x = -x

    angle = np.deg2rad(rotate_deg)
    c = np.cos(angle)
    s = np.sin(angle)

    xr = c * x - s * y
    yr = s * x + c * y

    return np.array([xr, yr], dtype=float)


def unique_centers_from_1_12(centers_slice, *, tol=1e-10):
    """
    Expand 1/12-slice centres to full hexagon centres.

    Strategy:
      1) original centre
      2) reflected centre across x=0
      3) rotate both by 0, 60, ..., 300 degrees
      4) remove duplicates
    """
    unique = {}

    for center in centers_slice:
        for reflect_x in [False, True]:
            for k in range(6):
                angle = 60.0 * k
                xy = transform_xy(center, reflect_x=reflect_x, rotate_deg=angle)

                key = (
                    int(np.round(xy[0] / tol)),
                    int(np.round(xy[1] / tol)),
                )

                unique[key] = xy

    centers = list(unique.values())

    # Stable ordering: center first, then increasing radius/angle.
    centers.sort(key=lambda p: (np.hypot(p[0], p[1]), np.arctan2(p[1], p[0])))

    return centers


def transform_mesh(mesh, *, reflect_x=False, rotate_deg=0.0):
    """
    Copy, reflect, and/or rotate a PyVista mesh.
    """
    new_mesh = mesh.copy(deep=True)

    if reflect_x:
        new_mesh = new_mesh.reflect(
            (1.0, 0.0, 0.0),
            point=(0.0, 0.0, 0.0),
            inplace=False,
        )

    if rotate_deg != 0.0:
        new_mesh.rotate_z(
            rotate_deg,
            point=(0.0, 0.0, 0.0),
            inplace=True,
        )

    return new_mesh


def expand_moderator_slice_to_full_hex(mod_grid, *, draw_outlines=True):
    """
    Expand the moderator 1/12 slice to a full hexagon.

    The moderator is expanded by reflection across x=0 and rotation by
    multiples of 60 degrees.
    """
    full_moderator_objects = []
    full_outline_objects = []

    sector_meshes = [
        mod_grid.copy(deep=True),
        transform_mesh(mod_grid, reflect_x=True),
    ]

    for k in range(6):
        angle = 60.0 * k

        for sector_mesh in sector_meshes:
            mesh_rot = transform_mesh(sector_mesh, rotate_deg=angle)

            full_moderator_objects.append(("moderator", mesh_rot))

            if draw_outlines:
                full_outline_objects.append(mesh_rot.extract_feature_edges())

    return full_moderator_objects, full_outline_objects


# -------------------------------------------------------------------------
# Build full-hex moderator
# -------------------------------------------------------------------------

body_objects = []
cap_objects = []

mod_grid = extrude_triangular_mesh_slice(
    points,
    triangles,
    z0_mid,
    z1_mid,
    T_mod_mid_plot,
    axial_scale=AXIAL_SCALE,
)

full_moderator_objects, full_outline_objects = expand_moderator_slice_to_full_hex(
    mod_grid,
    draw_outlines=True,
)

body_objects.extend(full_moderator_objects)


# -------------------------------------------------------------------------
# Build unique full-hex heat-pipe and fuel-pin centres
# -------------------------------------------------------------------------

HP_centers_full = unique_centers_from_1_12(HP_centers_slice)
fuel_pin_centers_full = unique_centers_from_1_12(fuel_pin_centers_slice)

print(f"Number of heat pipes rendered: {len(HP_centers_full)}")
print(f"Number of fuel pins rendered: {len(fuel_pin_centers_full)}")


# -------------------------------------------------------------------------
# Cap offset
# -------------------------------------------------------------------------

CAP_NUDGE = 0.003 * max(
    fp_z_edges[-1] - fp_z_edges[0],
    hp_z_edges[-1] - hp_z_edges[0],
) * AXIAL_SCALE

print(f"CAP_NUDGE = {CAP_NUDGE:.3e}")


# -------------------------------------------------------------------------
# Heat pipes
# -------------------------------------------------------------------------

for hp_center in HP_centers_full:
    theta0, theta1, n_theta = theta_settings_for_center(hp_center)

    # Wall/wick body
    hp_solid_grid = annular_sector_grid(
        center_xy=hp_center,
        r_edges=hp_r_edges_plot,
        z_edges=hp_z_edges,
        T_zr=T_solid_vap_full,
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
    )
    body_objects.append(("heat-pipe wall/wick", hp_solid_grid))

    # Wall/wick bottom cap
    hp_bottom_cap = annular_end_cap_grid(
        center_xy=hp_center,
        r_edges=hp_r_edges_plot,
        z=hp_z_edges[0],
        scalar_value=float(T_solid_vap_full[0, -1]),
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
        z_nudge=-CAP_NUDGE,
    )
    cap_objects.append(("heat-pipe wall/wick", hp_bottom_cap))

    # Wall/wick top cap
    hp_top_cap = annular_end_cap_grid(
        center_xy=hp_center,
        r_edges=hp_r_edges_plot,
        z=hp_z_edges[-1],
        scalar_value=float(T_solid_vap_full[-1, -1]),
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
        z_nudge=CAP_NUDGE,
    )
    cap_objects.append(("heat-pipe wall/wick", hp_top_cap))

    # Vapour body
    vapour_grid = annular_sector_grid(
        center_xy=hp_center,
        r_edges=vapour_r_edges,
        z_edges=hp_z_edges,
        T_zr=T_v_full[:, None],
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
    )
    body_objects.append(("heat-pipe vapour", vapour_grid))

    # Vapour bottom cap
    vap_bottom_cap = annular_end_cap_grid(
        center_xy=hp_center,
        r_edges=vapour_r_edges,
        z=hp_z_edges[0],
        scalar_value=float(T_v_full[0]),
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
        z_nudge=-1.5 * CAP_NUDGE,
    )
    cap_objects.append(("heat-pipe vapour", vap_bottom_cap))

    # Vapour top cap
    vap_top_cap = annular_end_cap_grid(
        center_xy=hp_center,
        r_edges=vapour_r_edges,
        z=hp_z_edges[-1],
        scalar_value=float(T_v_full[-1]),
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
        z_nudge=1.5 * CAP_NUDGE,
    )
    cap_objects.append(("heat-pipe vapour", vap_top_cap))


# -------------------------------------------------------------------------
# Fuel pins
# -------------------------------------------------------------------------

for fp_center in fuel_pin_centers_full:
    theta0, theta1, n_theta = theta_settings_for_center(fp_center)

    fp_grid = annular_sector_grid(
        center_xy=fp_center,
        r_edges=fp_r_edges_plot,
        z_edges=fp_z_edges,
        T_zr=T_FP_vap_full,
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
    )
    body_objects.append(("fuel pin", fp_grid))

    # Bottom cap: use the outer-wall temperature of the first axial cell
    fp_bottom_cap = annular_end_cap_grid(
        center_xy=fp_center,
        r_edges=fp_r_edges_plot,
        z=fp_z_edges[0],
        scalar_value=float(T_FP_vap_full[0, -1]),
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
        z_nudge=-CAP_NUDGE,
    )
    cap_objects.append(("fuel pin", fp_bottom_cap))

    # Top cap: use the outer-wall temperature of the last axial cell
    fp_top_cap = annular_end_cap_grid(
        center_xy=fp_center,
        r_edges=fp_r_edges_plot,
        z=fp_z_edges[-1],
        scalar_value=float(T_FP_vap_full[-1, -1]),
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
        z_nudge=CAP_NUDGE,
    )
    cap_objects.append(("fuel pin", fp_top_cap))


# -------------------------------------------------------------------------
# Render full hexagon
# Separate cmaps for:
#   1) fuel pin + moderator + heat-pipe wall/wick
#   2) heat-pipe vapour
# -------------------------------------------------------------------------

pv.set_jupyter_backend("none")
pv.OFF_SCREEN = False

SOLID_LABELS = {"fuel pin", "moderator", "heat-pipe wall/wick"}
VAPOUR_LABELS = {"heat-pipe vapour"}

SOLID_CMAP = "inferno"
VAPOUR_CMAP = "Blues"

full_objects = body_objects + cap_objects

solid_temperatures = np.concatenate([
    obj.cell_data["T"]
    for label, obj in full_objects
    if label in SOLID_LABELS
])

vapour_temperatures = np.concatenate([
    obj.cell_data["T"]
    for label, obj in full_objects
    if label in VAPOUR_LABELS
])

solid_clim = [
    float(np.nanmin(solid_temperatures)),
    float(np.nanmax(solid_temperatures)),
]

vapour_clim = [
    float(np.nanmin(vapour_temperatures)),
    float(np.nanmax(vapour_temperatures)),
]

print("Solid temperature range (FP + moderator + wall/wick):", solid_clim)
print("Vapour temperature range:", vapour_clim)

plotter = pv.Plotter(
    notebook=False,
    window_size=(1800, 1200),
)

plotter.set_background("white")
plotter.disable_depth_peeling()


# Main bodies first
for label, obj in body_objects:
    if label in VAPOUR_LABELS:
        cmap = VAPOUR_CMAP
        clim = vapour_clim
    else:
        cmap = SOLID_CMAP
        clim = solid_clim

    plotter.add_mesh(
        obj,
        scalars="T",
        clim=clim,
        cmap=cmap,
        opacity=1.0,
        show_edges=False,
        smooth_shading=True,
        show_scalar_bar=False,
        pickable=True,
        lighting=False,
    )


# End caps second, so they cover the original axial end faces
for label, obj in cap_objects:
    if label in VAPOUR_LABELS:
        cmap = VAPOUR_CMAP
        clim = vapour_clim
    else:
        cmap = SOLID_CMAP
        clim = solid_clim

    plotter.add_mesh(
        obj,
        scalars="T",
        clim=clim,
        cmap=cmap,
        opacity=1.0,
        show_edges=False,
        smooth_shading=False,
        show_scalar_bar=False,
        pickable=False,
        lighting=False,
    )


# Moderator outlines last
for outline in full_outline_objects:
    plotter.add_mesh(
        outline,
        color="black",
        line_width=0.35,
        opacity=1.0,
        show_scalar_bar=False,
        pickable=False,
    )


plotter.view_isometric()

try:
    plotter.show(
        interactive=True,
        auto_close=False,
    )
finally:
    plotter.close()
    pv.close_all()

Number of heat pipes rendered: 7
Number of fuel pins rendered: 24
CAP_NUDGE = 2.760e-04
Solid temperature range (FP + moderator + wall/wick): [994.890918946376, 1143.2730343555445]
Vapour temperature range: [1004.0800687658906, 1010.6603547283879]


MESA: error: ZINK: failed to choose pdev
glx: failed to create drisw screen
MESA: error: ZINK: failed to choose pdev
glx: failed to create drisw screen
MESA: error: ZINK: failed to choose pdev
glx: failed to create drisw screen


In [ ]:
import numpy as np
import pyvista as pv


# =============================================================================
# Full hexagon with one missing 1/6 wedge
# No artificial FP / wall-wick end caps
# Vapour axial end surfaces use the solid cmap
# Stronger component contours
# =============================================================================

# -------------------------------------------------------------------------
# Geometry layout matching create_hexagonal_reactor_mesh(...)
# -------------------------------------------------------------------------

pin_pitch = 0.032   # Must match the mesh-generation geometry
r_HP_mod  = 0.011   # Must match hex_mesh.msh
r_f_mod   = 0.0072  # Must match hex_mesh.msh

HP_centers_slice = [
    [0.0, 0.0],
    [np.sqrt(3.0) / 2.0 * pin_pitch, 1.5 * pin_pitch],
]

fuel_pin_centers_slice = [
    [0.0, 1.0 * pin_pitch],
    [0.0, 2.0 * pin_pitch],
    [np.sqrt(3.0) / 2.0 * pin_pitch, 2.5 * pin_pitch],
]


# -------------------------------------------------------------------------
# Plotting-only visual scaling
# -------------------------------------------------------------------------

AXIAL_SCALE = 0.04

FP_RADIAL_SCALE = r_f_mod / fp_r_edges[-1]

HP_RADIAL_SCALE = r_HP_mod / hp_r_edges[-1]
HEAT_PIPE_RADIUS_CLEARANCE = 0.995

fp_r_edges_plot = fp_r_edges * FP_RADIAL_SCALE
hp_r_edges_plot = hp_r_edges * HP_RADIAL_SCALE * HEAT_PIPE_RADIUS_CLEARANCE

VAPOUR_RADIUS_CLEARANCE = 0.995
vapour_radius_plot = hp_r_edges_plot[0] * VAPOUR_RADIUS_CLEARANCE
vapour_r_edges = np.array([0.0, vapour_radius_plot])


# -------------------------------------------------------------------------
# Missing 1/6 wedge control
# -------------------------------------------------------------------------

# Remove two adjacent 1/12 slices:
#   60--90 deg   -> centre 75 deg
#   90--120 deg  -> centre 105 deg
#
# Together this removes one full 1/6 sector:
#   60--120 deg -> centre 90 deg
MISSING_SLICE_CENTER_DEG = 135.0
MISSING_SLICE_WIDTH_DEG = 90.0

MISSING_A0 = np.deg2rad(MISSING_SLICE_CENTER_DEG - MISSING_SLICE_WIDTH_DEG / 2.0)
MISSING_A1 = np.deg2rad(MISSING_SLICE_CENTER_DEG + MISSING_SLICE_WIDTH_DEG / 2.0)

N_THETA_FULL = 180
N_THETA_CUT = 120


# -------------------------------------------------------------------------
# Contour controls
# -------------------------------------------------------------------------

DRAW_COMPONENT_CONTOURS = True
DRAW_MODERATOR_CONTOURS = True

COMPONENT_CONTOUR_WIDTH = 1.65
VAPOUR_CONTOUR_WIDTH = 2.55
MODERATOR_CONTOUR_WIDTH = 4.00

CONTOUR_COLOR = "black"

# -------------------------------------------------------------------------
# Moderator visual depth
# -------------------------------------------------------------------------

# Plotting-only depth multiplier for the moderator slice.
# 1.0 = actual one-cell Delta_Z thickness
# 3.0--8.0 usually gives a clearer 3D slab
MODERATOR_DEPTH_SCALE = 8.0


# -------------------------------------------------------------------------
# Optional moderator temperature shift
# -------------------------------------------------------------------------

T_mod_mid_plot = T_mod_mid.copy()
T_mod_mid_plot -= np.min(T_mod_mid_plot) - T_HP_outer_mid


# -------------------------------------------------------------------------
# Small geometry helpers
# -------------------------------------------------------------------------

def angle_difference_deg(a, b):
    return (a - b + 180.0) % 360.0 - 180.0


def cross2(a, b):
    return a[0] * b[1] - a[1] * b[0]


def transform_xy(point_xy, *, reflect_x=False, rotate_deg=0.0):
    x, y = float(point_xy[0]), float(point_xy[1])

    if reflect_x:
        x = -x

    angle = np.deg2rad(rotate_deg)
    c = np.cos(angle)
    s = np.sin(angle)

    xr = c * x - s * y
    yr = s * x + c * y

    return np.array([xr, yr], dtype=float)


def unique_centers_from_1_12(centers_slice, *, tol=1e-10):
    unique = {}

    for center in centers_slice:
        for reflect_x in [False, True]:
            for k in range(6):
                xy = transform_xy(
                    center,
                    reflect_x=reflect_x,
                    rotate_deg=60.0 * k,
                )

                key = (
                    int(np.round(xy[0] / tol)),
                    int(np.round(xy[1] / tol)),
                )

                unique[key] = xy

    centers = list(unique.values())
    centers.sort(key=lambda p: (np.hypot(p[0], p[1]), np.arctan2(p[1], p[0])))

    return centers


def transform_mesh(mesh, *, reflect_x=False, rotate_deg=0.0):
    new_mesh = mesh.copy(deep=True)

    if reflect_x:
        new_mesh = new_mesh.reflect(
            (1.0, 0.0, 0.0),
            point=(0.0, 0.0, 0.0),
            inplace=False,
        )

    if rotate_deg != 0.0:
        new_mesh.rotate_z(
            rotate_deg,
            point=(0.0, 0.0, 0.0),
            inplace=True,
        )

    return new_mesh


def make_polydata_from_quads(points, quads, values, name="T"):
    faces = []

    for q in quads:
        faces.extend([4, q[0], q[1], q[2], q[3]])

    poly = pv.PolyData(
        np.asarray(points, dtype=float),
        np.asarray(faces, dtype=np.int64),
    )

    poly.cell_data[name] = np.asarray(values, dtype=float)

    return poly


# -------------------------------------------------------------------------
# Vapour surface helpers
# -------------------------------------------------------------------------

def vapour_open_surface_grid(
    center_xy,
    r_edges,
    z_edges,
    T_z,
    *,
    theta0,
    theta1,
    n_theta,
    name="T",
    axial_scale=1.0,
):
    """
    Render the vapour body without axial end faces.

    This avoids z-fighting with separately rendered vapour axial end surfaces.
    The surface includes:
      - outer cylindrical surface
      - radial cut surfaces if the geometry is not 360 degrees
    """
    cx, cy = center_xy

    r_edges = np.asarray(r_edges, dtype=float)
    z_edges_plot = np.asarray(z_edges, dtype=float) * axial_scale
    T_z = np.asarray(T_z, dtype=float)

    theta_edges = np.linspace(theta0, theta1, n_theta)

    points_surf = []
    quads = []
    values = []

    n_z_edges = len(z_edges_plot)
    n_z_cells = n_z_edges - 1

    r_outer = r_edges[-1]

    # -------------------------------------------------------------
    # Outer cylindrical surface
    # -------------------------------------------------------------
    start = len(points_surf)

    for theta in theta_edges:
        for z in z_edges_plot:
            points_surf.append([
                cx + r_outer * np.cos(theta),
                cy + r_outer * np.sin(theta),
                z,
            ])

    for it in range(len(theta_edges) - 1):
        for iz in range(n_z_cells):
            p00 = start + it * n_z_edges + iz
            p01 = start + (it + 1) * n_z_edges + iz
            p11 = start + (it + 1) * n_z_edges + iz + 1
            p10 = start + it * n_z_edges + iz + 1

            quads.append([p00, p01, p11, p10])
            values.append(T_z[iz])

    # -------------------------------------------------------------
    # Cut surfaces for partial cylinders
    # -------------------------------------------------------------
    is_full_cylinder = np.isclose(
        np.mod(theta1 - theta0, 2.0 * np.pi),
        0.0,
        atol=1e-8,
    ) or np.isclose(theta1 - theta0, 2.0 * np.pi, atol=1e-8)

    if not is_full_cylinder:
        for theta in [theta0, theta1]:
            start = len(points_surf)

            for r in r_edges:
                for z in z_edges_plot:
                    points_surf.append([
                        cx + r * np.cos(theta),
                        cy + r * np.sin(theta),
                        z,
                    ])

            for ir in range(len(r_edges) - 1):
                for iz in range(n_z_cells):
                    p00 = start + ir * n_z_edges + iz
                    p01 = start + (ir + 1) * n_z_edges + iz
                    p11 = start + (ir + 1) * n_z_edges + iz + 1
                    p10 = start + ir * n_z_edges + iz + 1

                    quads.append([p00, p01, p11, p10])
                    values.append(T_z[iz])

    return make_polydata_from_quads(points_surf, quads, values, name=name)


def vapour_axial_end_surface_grid(
    center_xy,
    r_edges,
    z,
    scalar_value,
    *,
    theta0,
    theta1,
    n_theta,
    name="T",
    axial_scale=1.0,
):
    """
    Render one axial vapour end surface.

    This is not an overlay: the vapour body produced by
    vapour_open_surface_grid has no axial end faces.
    """
    cx, cy = center_xy

    r_edges = np.asarray(r_edges, dtype=float)
    theta_edges = np.linspace(theta0, theta1, n_theta)
    z_plot = z * axial_scale

    points_end = []
    quads = []
    values = []

    n_r = len(r_edges)

    for theta in theta_edges:
        for r in r_edges:
            points_end.append([
                cx + r * np.cos(theta),
                cy + r * np.sin(theta),
                z_plot,
            ])

    for it in range(len(theta_edges) - 1):
        for ir in range(n_r - 1):
            p00 = it * n_r + ir
            p01 = it * n_r + ir + 1
            p11 = (it + 1) * n_r + ir + 1
            p10 = (it + 1) * n_r + ir

            quads.append([p00, p01, p11, p10])
            values.append(scalar_value)

    return make_polydata_from_quads(points_end, quads, values, name=name)


# -------------------------------------------------------------------------
# Moderator expansion: 1/12 slice -> full hexagon with one 1/6 removed
# -------------------------------------------------------------------------

def expand_moderator_slice_to_full_hex_with_missing_slice(
    mod_grid,
    *,
    missing_slice_center_deg=90.0,
    missing_slice_width_deg=60.0,
    draw_outlines=True,
):
    """
    Expand the moderator 1/12 slice to a full hexagon, but omit all
    1/12 slices whose centres fall inside the removed angular wedge.

    For the current setup:
        missing_slice_center_deg = 90 deg
        missing_slice_width_deg  = 60 deg

    This removes:
        original  slice centre 75 deg
        reflected slice centre 105 deg

    i.e. one full 1/6 sector from 60 to 120 degrees.
    """
    full_moderator_objects = []
    full_outline_objects = []

    sector_meshes = [
        ("original", mod_grid.copy(deep=True), 75.0),
        ("reflected", transform_mesh(mod_grid, reflect_x=True), 105.0),
    ]

    half_width = missing_slice_width_deg / 2.0

    for k in range(6):
        rotation_angle = 60.0 * k

        for _, sector_mesh, base_center in sector_meshes:
            sector_center = (base_center + rotation_angle) % 360.0

            # Skip every 1/12 slice inside the removed 1/6 wedge.
            if abs(angle_difference_deg(sector_center, missing_slice_center_deg)) <= half_width + 1e-8:
                continue

            mesh_rot = transform_mesh(
                sector_mesh,
                rotate_deg=rotation_angle,
            )

            full_moderator_objects.append(("moderator", mesh_rot))

            if draw_outlines:
                full_outline_objects.append(mesh_rot.extract_feature_edges())

    return full_moderator_objects, full_outline_objects


# -------------------------------------------------------------------------
# Component opening logic near the missing wedge
# -------------------------------------------------------------------------

def theta_from_gap(gap_center, gap_width_deg, n_theta):
    gap_width = np.deg2rad(gap_width_deg)

    theta0 = gap_center + gap_width / 2.0
    theta1 = gap_center + 2.0 * np.pi - gap_width / 2.0

    return theta0, theta1, n_theta


def component_missing_slice_status(center_xy, radius):
    p = np.asarray(center_xy, dtype=float)

    if np.linalg.norm(p) < 1e-12:
        return "intersect"

    u0 = np.array([np.cos(MISSING_A0), np.sin(MISSING_A0)])
    u1 = np.array([np.cos(MISSING_A1), np.sin(MISSING_A1)])

    d0 = cross2(u0, p)
    d1 = -cross2(u1, p)

    if d0 > radius and d1 > radius:
        return "inside"

    if d0 < -radius or d1 < -radius:
        return "outside"

    return "intersect"


def theta_settings_for_missing_slice(center_xy, radius):
    status = component_missing_slice_status(center_xy, radius)

    if status == "inside":
        return None

    if status == "outside":
        return 0.0, 2.0 * np.pi, N_THETA_FULL

    p = np.asarray(center_xy, dtype=float)

    if np.linalg.norm(p) < 1e-12:
        return MISSING_A1, MISSING_A0 + 2.0 * np.pi, N_THETA_CUT

    u0 = np.array([np.cos(MISSING_A0), np.sin(MISSING_A0)])
    u1 = np.array([np.cos(MISSING_A1), np.sin(MISSING_A1)])

    d0 = cross2(u0, p)
    d1 = -cross2(u1, p)

    if abs(d0) <= abs(d1):
        gap_center = MISSING_A0 + np.pi / 2.0
    else:
        gap_center = MISSING_A1 - np.pi / 2.0

    return theta_from_gap(
        gap_center=gap_center,
        gap_width_deg=180.0,
        n_theta=N_THETA_CUT,
    )


def safe_feature_edges(mesh, feature_angle=25.0):
    """
    Extract strong visual contours without showing every mesh cell edge.
    """
    try:
        edges = mesh.extract_feature_edges(
            feature_angle=feature_angle,
            boundary_edges=True,
            feature_edges=True,
            manifold_edges=False,
            non_manifold_edges=False,
        )
        return edges
    except Exception:
        return None


# -------------------------------------------------------------------------
# Build full-hex moderator with missing 1/6 wedge
# -------------------------------------------------------------------------

body_objects = []
vapour_end_objects = []

# -------------------------------------------------------------------------
# Build visually thickened moderator slab
# -------------------------------------------------------------------------

z_mid_mod = 0.5 * (z0_mid + z1_mid)
dz_mod = (z1_mid - z0_mid) * MODERATOR_DEPTH_SCALE

z0_mod_plot = z_mid_mod - 0.5 * dz_mod
z1_mod_plot = z_mid_mod + 0.5 * dz_mod

mod_grid = extrude_triangular_mesh_slice(
    points,
    triangles,
    z0_mod_plot,
    z1_mod_plot,
    T_mod_mid_plot,
    axial_scale=AXIAL_SCALE,
)

full_moderator_objects, full_outline_objects = (
    expand_moderator_slice_to_full_hex_with_missing_slice(
        mod_grid,
        missing_slice_center_deg=MISSING_SLICE_CENTER_DEG,
        missing_slice_width_deg=MISSING_SLICE_WIDTH_DEG,
        draw_outlines=True,
    )
)

body_objects.extend(full_moderator_objects)


# -------------------------------------------------------------------------
# Build unique full-hex heat-pipe and fuel-pin centres
# -------------------------------------------------------------------------

HP_centers_full = unique_centers_from_1_12(HP_centers_slice)
fuel_pin_centers_full = unique_centers_from_1_12(fuel_pin_centers_slice)

print(f"Number of heat pipes before wedge removal: {len(HP_centers_full)}")
print(f"Number of fuel pins before wedge removal: {len(fuel_pin_centers_full)}")


# -------------------------------------------------------------------------
# Heat pipes
# -------------------------------------------------------------------------

n_hp_rendered = 0

for hp_center in HP_centers_full:
    theta_settings = theta_settings_for_missing_slice(
        hp_center,
        radius=r_HP_mod,
    )

    if theta_settings is None:
        continue

    theta0, theta1, n_theta = theta_settings
    n_hp_rendered += 1

    # Wall/wick body: no artificial end-cap overlays.
    hp_solid_grid = annular_sector_grid(
        center_xy=hp_center,
        r_edges=hp_r_edges_plot,
        z_edges=hp_z_edges,
        T_zr=T_solid_vap_full,
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
    )
    body_objects.append(("heat-pipe wall/wick", hp_solid_grid))

    # Vapour body: open at axial ends.
    vapour_grid = vapour_open_surface_grid(
        center_xy=hp_center,
        r_edges=vapour_r_edges,
        z_edges=hp_z_edges,
        T_z=T_v_full,
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
    )
    body_objects.append(("heat-pipe vapour", vapour_grid))

    # Vapour axial end surfaces use the solid cmap.
    # These do not overlap with the vapour body, because the vapour body
    # has no axial end faces.
    vap_bottom_end = vapour_axial_end_surface_grid(
        center_xy=hp_center,
        r_edges=vapour_r_edges,
        z=hp_z_edges[0],
        scalar_value=float(T_v_full[0]),
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
    )
    vapour_end_objects.append(("heat-pipe vapour end", vap_bottom_end))

    vap_top_end = vapour_axial_end_surface_grid(
        center_xy=hp_center,
        r_edges=vapour_r_edges,
        z=hp_z_edges[-1],
        scalar_value=float(T_v_full[-1]),
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
    )
    vapour_end_objects.append(("heat-pipe vapour end", vap_top_end))


# -------------------------------------------------------------------------
# Fuel pins
# -------------------------------------------------------------------------

n_fp_rendered = 0

for fp_center in fuel_pin_centers_full:
    theta_settings = theta_settings_for_missing_slice(
        fp_center,
        radius=r_f_mod,
    )

    if theta_settings is None:
        continue

    theta0, theta1, n_theta = theta_settings
    n_fp_rendered += 1

    # Fuel-pin body: no artificial end-cap overlays.
    fp_grid = annular_sector_grid(
        center_xy=fp_center,
        r_edges=fp_r_edges_plot,
        z_edges=fp_z_edges,
        T_zr=T_FP_vap_full,
        theta0=theta0,
        theta1=theta1,
        n_theta=n_theta,
        axial_scale=AXIAL_SCALE,
    )
    body_objects.append(("fuel pin", fp_grid))


print(f"Number of heat pipes rendered after wedge removal: {n_hp_rendered}")
print(f"Number of fuel pins rendered after wedge removal: {n_fp_rendered}")


# -------------------------------------------------------------------------
# Render full hexagon with missing 1/6 wedge
# Separate cmaps for:
#   1) fuel pin + moderator + wall/wick + vapour axial ends
#   2) heat-pipe vapour body
# -------------------------------------------------------------------------

pv.set_jupyter_backend("none")
pv.OFF_SCREEN = False

SOLID_LABELS = {
    "fuel pin",
    "moderator",
    "heat-pipe wall/wick",
    "heat-pipe vapour end",
}

VAPOUR_LABELS = {
    "heat-pipe vapour",
}

SOLID_CMAP = "afmhot"
VAPOUR_CMAP = "Blues"

full_objects = body_objects + vapour_end_objects

solid_temperatures = np.concatenate([
    obj.cell_data["T"]
    for label, obj in full_objects
    if label in SOLID_LABELS
])

vapour_temperatures = np.concatenate([
    obj.cell_data["T"]
    for label, obj in full_objects
    if label in VAPOUR_LABELS
])

solid_clim = [
    float(np.nanmin(solid_temperatures)) - 20.,
    float(np.nanmax(solid_temperatures)),
]

vapour_clim = [
    float(np.nanmin(vapour_temperatures)),
    float(np.nanmax(vapour_temperatures)),
]

print("Solid temperature range (FP + moderator + wall/wick + vapour ends):", solid_clim)
print("Vapour body temperature range:", vapour_clim)

plotter = pv.Plotter(
    notebook=False,
    window_size=(1800, 1200),
)

plotter.set_background("white")
plotter.disable_depth_peeling()


# -------------------------------------------------------------------------
# Main temperature surfaces
# -------------------------------------------------------------------------

for label, obj in body_objects:
    if label in VAPOUR_LABELS:
        cmap = VAPOUR_CMAP
        clim = vapour_clim
    else:
        cmap = SOLID_CMAP
        clim = solid_clim

    plotter.add_mesh(
        obj,
        scalars="T",
        clim=clim,
        cmap=cmap,
        opacity=1.0,
        show_edges=False,
        smooth_shading=True,
        show_scalar_bar=False,
        pickable=True,
        lighting=False,
    )


# Vapour axial end surfaces: solid cmap
for label, obj in vapour_end_objects:
    plotter.add_mesh(
        obj,
        scalars="T",
        clim=solid_clim,
        cmap=SOLID_CMAP,
        opacity=1.0,
        show_edges=False,
        smooth_shading=False,
        show_scalar_bar=False,
        pickable=False,
        lighting=False,
    )


# -------------------------------------------------------------------------
# Stronger contours / shape outlines
# -------------------------------------------------------------------------

# Moderator slice outlines
if DRAW_MODERATOR_CONTOURS:
    for outline in full_outline_objects:
        plotter.add_mesh(
            outline,
            color=CONTOUR_COLOR,
            line_width=MODERATOR_CONTOUR_WIDTH,
            opacity=1.0,
            show_scalar_bar=False,
            pickable=False,
        )

# Component contours
if DRAW_COMPONENT_CONTOURS:
    component_objects_for_contours = [
        (label, obj)
        for label, obj in body_objects
        if label != "moderator"
    ] + vapour_end_objects

    for label, obj in component_objects_for_contours:
        edges = safe_feature_edges(obj, feature_angle=25.0)

        if edges is not None and edges.n_cells > 0:
            if label == "heat-pipe vapour":
                line_width = VAPOUR_CONTOUR_WIDTH
            else:
                line_width = COMPONENT_CONTOUR_WIDTH

            plotter.add_mesh(
                edges,
                color=CONTOUR_COLOR,
                line_width=line_width,
                opacity=1.0,
                show_scalar_bar=False,
                pickable=False,
            )


# -------------------------------------------------------------------------
# Camera / export controls
# -------------------------------------------------------------------------

from pathlib import Path
from PIL import Image

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

png_path = output_dir / "fuel_assembly_temperature_render_1_6_wedge.png"
pdf_path = output_dir / "fuel_assembly_temperature_render_1_6_wedge.pdf"
camera_path = output_dir / "fuel_assembly_camera_position_1_6_wedge.txt"

# Force native VTK render window.
pv.set_jupyter_backend("none")
pv.OFF_SCREEN = False

plotter.view_isometric()

print("1. Rotate / zoom / pan the PyVista window.")
print("2. Press 'q' in the PyVista window when the view is correct.")
print("3. The figure will be saved after the window closes.")

# Let you interactively choose the camera.
# Important: use 'q' to close the window, not the X button.
plotter.show(
    interactive=True,
    auto_close=False,
)

# Save the camera position after interaction.
with open(camera_path, "w") as f:
    f.write(repr(plotter.camera_position))

# Render once more before screenshot.
plotter.render()

# Save screenshot.
# Use image_scale on the plotter instead of screenshot(scale=...),
# which is more version-stable.
plotter.image_scale = 4
plotter.screenshot(
    str(png_path),
    transparent_background=False,
)

# Convert PNG to PDF.
img = Image.open(png_path).convert("RGB")
img.save(
    pdf_path,
    "PDF",
    resolution=300.0,
)

print(f"Saved PNG: {png_path.resolve()}")
print(f"Saved PDF: {pdf_path.resolve()}")
print(f"Saved camera position: {camera_path.resolve()}")
print("Camera position:")
print(plotter.camera_position)

plotter.close()
pv.close_all()

Number of heat pipes before wedge removal: 7
Number of fuel pins before wedge removal: 24
Number of heat pipes rendered after wedge removal: 6
Number of fuel pins rendered after wedge removal: 19
Solid temperature range (FP + moderator + wall/wick + vapour ends): [979.890918946376, 1143.2730343555445]
Vapour body temperature range: [1004.0800687658906, 1010.6603547283879]


MESA: error: ZINK: failed to choose pdev
glx: failed to create drisw screen
MESA: error: ZINK: failed to choose pdev
glx: failed to create drisw screen


1. Rotate / zoom / pan the PyVista window.
2. Press 'q' in the PyVista window when the view is correct.
3. The figure will be saved after the window closes.


MESA: error: ZINK: failed to choose pdev
glx: failed to create drisw screen


Saved PNG: /home/felixpersson/MasterThesisProject/outputs/fuel_assembly_temperature_render_1_6_wedge.png
Saved PDF: /home/felixpersson/MasterThesisProject/outputs/fuel_assembly_temperature_render_1_6_wedge.pdf
Saved camera position: /home/felixpersson/MasterThesisProject/outputs/fuel_assembly_camera_position_1_6_wedge.txt
Camera position:
[(-0.08723524960807205, 0.12210972361428368, 0.43933747020563263),
 (0.0003043549797796183, 0.0016495119383498277, 0.0455630558096689),
 (-0.8676712576376004, -0.4954168076679011, -0.04133733602605441)]


True